# Лабораторна робота №4
## Моніторинг, відстеження та версіонування ML-експериментів за допомогою MLflow

**Частина 1.** Локальне використання MLflow (SQLite backend + локальний artifact store) для логування
параметрів, метрик, тегів та артефактів навчання YOLO у Jupyter Notebook.

## 1. Навчання моделі YOLO (як у ЛР3)

In [ ]:
import os
from pathlib import Path
from ultralytics import YOLO
import ultralytics
from ultralytics import settings

os.chdir('/home/student/projects/mlops-yolo-detection')
ultralytics.settings.update({'datasets_dir': '/'})

DATA_PATH = Path('datasets/construction-ppe-small/data.yaml').resolve()
PROJECT_DIR = Path('runs/lab04_notebook').resolve()
EXPERIMENT_NAME = 'yolo11n_construction_ppe_small_mlflow'

print('CWD:', os.getcwd())
print('Dataset config exists:', DATA_PATH.exists())

# Вимикаємо автоматичну інтеграцію Ultralytics з MLflow
settings.update({'mlflow': False})

model = YOLO('yolo11n.pt')
results = model.train(data=str(DATA_PATH), epochs=10, imgsz=320, batch=4,
                      device='cpu', workers=0,
                      project=PROJECT_DIR, name=EXPERIMENT_NAME, exist_ok=True)
run_dir = Path(results.save_dir)
print('Run directory:', run_dir)

In [ ]:
import pandas as pd
results_csv = run_dir / 'results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()
df.head()

## 2. Налаштування локального MLflow (SQLite + локальний artifact store)

In [ ]:
import mlflow

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

MLFLOW_DIR = ROOT / 'data' / 'mlflow-local'
MLFLOW_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = MLFLOW_DIR / 'mlflow.db'
ARTIFACT_PATH = MLFLOW_DIR / 'artifacts'
ARTIFACT_PATH.mkdir(exist_ok=True)

mlflow.set_tracking_uri(f'sqlite:///{DB_PATH.as_posix()}')

EXPERIMENT = 'Construction-PPE-YOLO-Notebook'
if mlflow.get_experiment_by_name(EXPERIMENT) is None:
    mlflow.create_experiment(name=EXPERIMENT, artifact_location=ARTIFACT_PATH.as_uri())
mlflow.set_experiment(EXPERIMENT)
print('Tracking URI:', mlflow.get_tracking_uri())

## 3. Версія вихідного коду з Git

In [ ]:
import subprocess

def git_info(*args):
    r = subprocess.run(['git', *args], cwd=ROOT, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else 'unknown'

GIT_COMMIT = git_info('rev-parse', 'HEAD')
GIT_BRANCH = git_info('branch', '--show-current')
GIT_DIRTY = bool(git_info('status', '--porcelain'))
print('commit:', GIT_COMMIT[:10], '| branch:', GIT_BRANCH, '| dirty:', GIT_DIRTY)

## 4. Реєстрація параметрів і тегів (MLflow Run)

In [ ]:
with mlflow.start_run(run_name='yolo11n-notebook-01') as run:
    mlflow.log_params({'model': 'yolo11n.pt', 'epochs': 10, 'imgsz': 320, 'batch': 4})
    mlflow.set_tags({
        'dataset.name': 'Construction-PPE-small',
        'dataset.revision': GIT_COMMIT,
        'git.commit': GIT_COMMIT,
        'git.branch': GIT_BRANCH,
        'git.dirty': str(GIT_DIRTY).lower(),
        'model.family': 'YOLO11',
        'task': 'object_detection',
        'environment': 'jupyter',
    })
    MLFLOW_RUN_ID = run.info.run_id
print('MLflow Run ID:', MLFLOW_RUN_ID)

## 5. Реєстрація метрик за епохами

In [ ]:
import math

metric_prefixes = {
    'precision': 'metrics/precision',
    'recall': 'metrics/recall',
    'mAP50': 'metrics/mAP50(',
    'mAP50_95': 'metrics/mAP50-95',
}
metric_columns = {}
for metric_name, prefix in metric_prefixes.items():
    matches = [c for c in df.columns if c.lower().startswith(prefix.lower())]
    if not matches:
        raise ValueError(f'Metric column not found: {metric_name}')
    metric_columns[metric_name] = matches[0]

with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    for index, row in df.iterrows():
        epoch = int(row['epoch']) if 'epoch' in df.columns else index + 1
        for metric_name, column in metric_columns.items():
            value = float(row[column])
            if math.isfinite(value):
                mlflow.log_metric(metric_name, value, step=epoch)
print('Metrics logged per epoch')

## 6. Реєстрація артефактів (ваги, графіки, конфігурації)

In [ ]:
with mlflow.start_run(run_id=MLFLOW_RUN_ID):
    files = {
        'results.csv': 'metrics',
        'results.png': 'plots',
        'confusion_matrix.png': 'plots',
        'args.yaml': 'metadata',
    }
    for filename, destination in files.items():
        path = run_dir / filename
        if path.is_file():
            mlflow.log_artifact(str(path), artifact_path=destination)

    best_model = run_dir / 'weights' / 'best.pt'
    if best_model.is_file():
        mlflow.log_artifact(str(best_model), artifact_path='model')
print('Artifacts logged')

## 7. Перегляд результатів у MLflow UI

В окремому терміналі (той самий шлях до SQLite):
```bash
mlflow ui --backend-store-uri "sqlite:///$PWD/data/mlflow-local/mlflow.db" --host 127.0.0.1 --port 5000
```
Через SSH-тунель `ssh -N -L 5000:127.0.0.1:5000 ...` відкрити http://localhost:5000, знайти експеримент
`Construction-PPE-YOLO-Notebook`, переглянути параметри, метрики, їх динаміку, теги та артефакти.
Другий Run (інші гіперпараметри) логуеться аналогічно та порівнюється у MLflow UI.